# SMARTBind Inference script on 10-fold random-split HARIBOSS with Decoyfinder- and DeepCoy-generated Decoys

In [2]:
import warnings

warnings.filterwarnings("ignore")
import sys
sys.path.append("../../../..")  
from smartbind import load_smartbind_models
from smartbind import logger
from smartbind.preprocess import convert_smiles_to_pf2

import pandas as pd
from torch.nn.functional import cosine_similarity
import pickle
import tqdm

In [3]:
device = 'cpu'
# load decoys
with open('data/decoyfinder_decoys/decoyfinder_decoys.pkl', 'rb') as f:
    decoy_smiles_dict = pickle.load(f)

all_scores = []
for fold in tqdm.tqdm(range(1, 11)):
    print(f'Processing fold {fold}')
    logger.info('Get SmartBind pre-trained model objects')
    smartbind_model = load_smartbind_models(
        model_path=f'../../../../SMARTBind_weight/fold{fold}.pth',
        device=device,
        vs_mode=True
    )
    val_pd = pd.read_csv(f'data/10fd_random_smartbind_format/val_fold_{fold}.csv')
    # calculate the rank percentile for each
    for i in range(len(val_pd)):
        ligand_smiles = val_pd.iloc[i]['SMILES']
        target_id = val_pd.iloc[i]['Ligand ID']
        try:
            decoy_smiles_list = decoy_smiles_dict[target_id]
        except KeyError:
            # print(f'No decoys found for ligand {target_id}, skipping...')
            continue
        all_smiles = [ligand_smiles] + decoy_smiles_list
        all_fp2s = [convert_smiles_to_pf2(smiles) for smiles in all_smiles]
        rna = val_pd.iloc[i]['RNA']

        smol_embeds = smartbind_model.inference_list_smols(all_fp2s)
        rna_embed = smartbind_model.inference_single_rna(rna)
        similarities = cosine_similarity(rna_embed, smol_embeds).tolist()
        ligand_score = similarities[0]
        decoy_scores = similarities[1:]
        # Calculate rank percentile
        rank = 0
        for decoy_score in decoy_scores:
            if ligand_score >= decoy_score:
                rank += 1
        rank_percentile = (len(decoy_scores) + 1 - rank) / (len(decoy_scores) + 1)
        all_scores.append(1 - rank_percentile)

print(f'Average all_scores: {sum(all_scores)/len(all_scores):.4f}')
print(f'Median all_scores: {sorted(all_scores)[len(all_scores) // 2]:.4f}')
print(f'Standard deviation of all_scores: {pd.Series(all_scores).std():.4f}')
# save in pkl
with open('smartbind_decoyfinder_no_leakage_update.pkl', 'wb') as f:
    pickle.dump(all_scores, f)

  0%|          | 0/10 [00:00<?, ?it/s]

Processing fold 1
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42


 10%|█         | 1/10 [00:25<03:50, 25.64s/it]

Processing fold 2
SMARTBind - INFO - Get SmartBind pre-trained model objects


...

Processing fold 9
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42
 90%|█████████ | 9/10 [04:02<00:27, 27.87s/it]

Processing fold 10
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42
100%|██████████| 10/10 [04:26<00:00, 26.62s/it]

Average all_scores: 0.8951
Median all_scores: 0.9901
Standard deviation of all_scores: 0.2426


In [4]:
# load decoys
with open('data/deepcoy_decoys/molecules_ligands_DUDE_smiles.pickle', 'rb') as f:
    decoy_smiles_dict = pickle.load(f)

all_scores = []
for fold in tqdm.tqdm(range(1, 11)):
    print(f'Processing fold {fold}')
    logger.info('Get SmartBind pre-trained model objects')
    smartbind_model = load_smartbind_models(
        model_path=f'../../../../SMARTBind_weight/fold{fold}.pth',
        device=device,
        vs_mode=True
    )
    val_pd = pd.read_csv(f'data/10fd_random_smartbind_format/val_fold_{fold}.csv')
    # calculate the rank percentile for each
    for i in range(len(val_pd)):
        ligand_smiles = val_pd.iloc[i]['SMILES']
        target_id = val_pd.iloc[i]['Ligand ID']
        try:
            decoy_smiles_list = decoy_smiles_dict[target_id]
        except KeyError:
            # print(f'No decoys found for ligand {target_id}, skipping...')
            continue
        all_smiles = [ligand_smiles] + decoy_smiles_list
        all_fp2s = [convert_smiles_to_pf2(smiles) for smiles in all_smiles]
        rna = val_pd.iloc[i]['RNA']

        smol_embeds = smartbind_model.inference_list_smols(all_fp2s)
        rna_embed = smartbind_model.inference_single_rna(rna)
        similarities = cosine_similarity(rna_embed, smol_embeds).tolist()
        ligand_score = similarities[0]
        decoy_scores = similarities[1:]
        # Calculate rank percentile
        rank = 0
        for decoy_score in decoy_scores:
            if ligand_score >= decoy_score:
                rank += 1
        rank_percentile = (len(decoy_scores) + 1 - rank) / (len(decoy_scores) + 1)
        all_scores.append(1 - rank_percentile)

print(f'Average all_scores: {sum(all_scores)/len(all_scores):.4f}')
print(f'Median all_scores: {sorted(all_scores)[len(all_scores) // 2]:.4f}')
print(f'Standard deviation of all_scores: {pd.Series(all_scores).std():.4f}')
# save in pkl
with open('smartbind_deepcoy_dude_no_leakage_update.pkl', 'wb') as f:
    pickle.dump(all_scores, f)

  0%|          | 0/10 [00:00<?, ?it/s]

Processing fold 1
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42


 10%|█         | 1/10 [01:03<09:31, 63.51s/it]

Processing fold 2
SMARTBind - INFO - Get SmartBind pre-trained model objects


...

Processing fold 9
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42
*** Open Babel Warning  in ParseSmiles
  Failed to kekulize aromatic SMILES

 90%|█████████ | 9/10 [09:05<01:00, 60.55s/it]

Processing fold 10
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42
100%|██████████| 10/10 [09:59<00:00, 59.98s/it]

Average all_scores: 0.9101
Median all_scores: 0.9980
Standard deviation of all_scores: 0.2310


In [5]:
device = 'cpu'
# load decoys
with open('data/deepcoy_decoys/molecules_ligands_DEKOIS_smiles.pickle', 'rb') as f:
    decoy_smiles_dict = pickle.load(f)

all_scores = []
for fold in tqdm.tqdm(range(1, 11)):
    print(f'Processing fold {fold}')
    logger.info('Get SmartBind pre-trained model objects')
    smartbind_model = load_smartbind_models(
        model_path=f'../../../../SMARTBind_weight/fold{fold}.pth',
        device=device,
        vs_mode=True
    )
    val_pd = pd.read_csv(f'data/10fd_random_smartbind_format/val_fold_{fold}.csv')
    # calculate the rank percentile for each
    for i in range(len(val_pd)):
        ligand_smiles = val_pd.iloc[i]['SMILES']
        target_id = val_pd.iloc[i]['Ligand ID']
        try:
            decoy_smiles_list = decoy_smiles_dict[target_id]
        except KeyError:
            # print(f'No decoys found for ligand {target_id}, skipping...')
            continue
        all_smiles = [ligand_smiles] + decoy_smiles_list
        all_fp2s = [convert_smiles_to_pf2(smiles) for smiles in all_smiles]
        rna = val_pd.iloc[i]['RNA']

        smol_embeds = smartbind_model.inference_list_smols(all_fp2s)
        rna_embed = smartbind_model.inference_single_rna(rna)
        similarities = cosine_similarity(rna_embed, smol_embeds).tolist()
        ligand_score = similarities[0]
        decoy_scores = similarities[1:]
        # Calculate rank percentile
        rank = 0
        for decoy_score in decoy_scores:
            if ligand_score >= decoy_score:
                rank += 1
        rank_percentile = (len(decoy_scores) + 1 - rank) / (len(decoy_scores) + 1)
        all_scores.append(1 - rank_percentile)

print(f'Average all_scores: {sum(all_scores)/len(all_scores):.4f}')
print(f'Median all_scores: {sorted(all_scores)[len(all_scores) // 2]:.4f}')
print(f'Standard deviation of all_scores: {pd.Series(all_scores).std():.4f}')
# save in pkl
with open('smartbind_deepcoy_dekois_no_leakage_update.pkl', 'wb') as f:
    pickle.dump(all_scores, f)

  0%|          | 0/10 [00:00<?, ?it/s]

Processing fold 1
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42


*** Open Babel Warning  in ParseSmiles
  Failed to kekulize aromatic SMILES

 10%|█         | 1/10 [01:03<09:27, 63.02s/it]

Processing fold 2
SMARTBind - INFO - Get SmartBind pre-trained model objects


...

Processing fold 9
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42
*** Open Babel Warning  in ParseSmiles
  Failed to kekulize aromatic SMILES

 90%|█████████ | 9/10 [09:11<01:00, 60.06s/it]

Processing fold 10
SMARTBind - INFO - Get SmartBind pre-trained model objects


Global seed set to 42
100%|██████████| 10/10 [10:07<00:00, 60.73s/it]

Average all_scores: 0.9093
Median all_scores: 0.9980
Standard deviation of all_scores: 0.2305
